# Reward-model approach

Instead of training a policy with TD3, learn a **reward model**: a network mapping
`(state, action)` to predicted reward. The controller is then `argmax` over candidate
actions.

Why this instead of TD3:

- **No exploration failure.** Actions are sampled uniformly by us, so the training data
  covers the whole space. TD3's untrained actor only ever produced 1600-3500 uA/cm2, which
  is why it never found the good low-amplitude region.
- **MATLAB runs once per severity, not once per architecture.** The same dataset trains
  32, 64, or any other size.
- **Fitting takes seconds.** Comparing architectures becomes a supervised-learning
  question, not a 14-hour experiment.

| # | Step | MATLAB | Time |
|---|---|---|---|
| 1 | Setup | no | instant |
| 2 | Start MATLAB | yes | ~1 min |
| 3 | Collect data (3 severities) | yes | ~45-70 min |
| 4 | Fit all architectures | no | ~1 min |
| 5 | Cross-evaluation matrix | no | instant |
| 6 | Which action does each pick | no | instant |
| 7 | Validate best action in MATLAB | yes | ~10 min |

## 1. Setup

In [1]:
import importlib
import json
import numpy as np
from pathlib import Path
from dataclasses import replace

import dbs_core, dbs_checks, dbs_plots, dbs_reward_model as rm
for m in (dbs_core, dbs_checks, dbs_plots, rm):
    importlib.reload(m)

from dbs_core import Config, MatlabEnv, MockEnv

assert dbs_checks.versions()
print("reward model module:", rm.__version__)

module versions:
  dbs_core     2.6.0
  dbs_plots    2.6.0
  dbs_checks   2.6.0

  all modules at 2.6.0 — files are in sync
reward model module: 2.6.0


## 2. Start MATLAB

In [2]:
cfg = Config(
    pd_value=0.4,
    tmax=1500,
    steps_per_episode=15,
    rms_max=1200.0,
    run_dir="runs",
    tag="rm",
)
menv = MatlabEnv(cfg, mat_path="bgn_vars.mat")
print("engine started")

engine started


## 3. Collect

Uniform random actions, each held for `hold` steps so the biomarker settles before the
sample is recorded (the probe data showed ~3 steps of turn-on transient).

`n_samples=400, hold=4` is about 1600 MATLAB steps per severity — roughly 23 min each at
0.88 s/step. Raw `sgis` and `rms` are stored, so rewards can be recomputed under different
weights without re-collecting.

In [ ]:
PD_VALUES = [0.3, 0.6, 1.0]
N_SAMPLES = 400
HOLD = 4

datasets = {}
for pd in PD_VALUES:
    c = replace(cfg, pd_value=pd)
    menv.cfg = c
    ok, samples, _ = dbs_checks.live(menv, c, verbose=False)
    print(f"pd={pd}: env checks {'PASS' if ok else 'FAIL — treat with caution'}")
    datasets[f"pd{pd}"] = rm.collect(menv, c, n_samples=N_SAMPLES, hold=HOLD)

  [PASS] reset returns correct state_dim
  [PASS] state values finite
  [PASS] hjorth window has enough samples — window (10, 250), expected ~250 (sim_time_per_step=100 ms / dt=0.01)

  [PASS] sgis in plausible range — median 452.1, expected order 100-2000
  [PASS] feature 'sd' inside bounds — 28/28 in [25.125, 34.542], observed [26.640, 32.875]
  [PASS] feature 'activity' inside bounds — 28/28 in [634.501, 1189.581], observed [713.278, 1081.517]
  [PASS] feature 'mobility' inside bounds — 28/28 in [0.807, 0.910], observed [0.819, 0.900]
  [PASS] feature 'complexity' inside bounds — 28/28 in [1.505, 1.683], observed [1.514, 1.651]
  [PASS] biomarker term not saturated — mean r1 = 0.132
  [PASS] reward varies across actions — spread 0.1514
  [PASS] stimulating beats doing nothing — margin +0.1168 — below 0.02 risks no-stim collapse

live environment checks: 11 passed, 0 failed
pd=0.3: env checks PASS
collecting 400 samples at pd=0.3 (hold=4, 134 episodes, ~1608 MATLAB steps)
  15/400 sa

In [3]:
PD_VALUES = [0.3, 0.6, 1.0]
N_SAMPLES = 400
HOLD = 4

c = replace(cfg, pd_value=1.0)
menv.cfg = c
ok, samples, _ = dbs_checks.live(menv, c, verbose=False)
print(f"pd=1.0: env checks {'PASS' if ok else 'FAIL'}")
rm.collect(menv, c, n_samples=N_SAMPLES, hold=HOLD)

  [PASS] reset returns correct state_dim
  [PASS] state values finite
  [PASS] hjorth window has enough samples — window (10, 250), expected ~250 (sim_time_per_step=100 ms / dt=0.01)

  [PASS] sgis in plausible range — median 615.3, expected order 100-2000
  [PASS] feature 'sd' inside bounds — 28/28 in [25.125, 34.542], observed [27.640, 32.786]
  [PASS] feature 'activity' inside bounds — 28/28 in [634.501, 1189.581], observed [770.491, 1075.180]
  [PASS] feature 'mobility' inside bounds — 28/28 in [0.807, 0.910], observed [0.813, 0.889]
  [PASS] feature 'complexity' inside bounds — 28/28 in [1.505, 1.683], observed [1.536, 1.658]
  [PASS] biomarker term not saturated — mean r1 = 0.212
  [PASS] reward varies across actions — spread 0.2132
  [PASS] stimulating beats doing nothing — margin +0.0818 — below 0.02 risks no-stim collapse

live environment checks: 11 passed, 0 failed
pd=1.0: env checks PASS
collecting 400 samples at pd=1.0 (hold=4, 134 episodes, ~1608 MATLAB steps)
  15/400 sa

[{'pd': 1.0,
  's0': 0.5637608766555786,
  's1': 0.5295236706733704,
  's2': 0.2330676019191742,
  's3': 0.6753222346305847,
  'a_freq_norm': 0.0976270078546495,
  'a_amp_norm': 0.43037873274483895,
  'freq': 101.53049822655508,
  'amp': 3575.9468318620975,
  'sgis': 937.2950692586751,
  'rms': 619.3721597950114},
 {'pd': 1.0,
  's0': 0.5786740779876709,
  's1': 0.5428946614265442,
  's2': 0.4987969398498535,
  's3': 0.5038391947746277,
  'a_freq_norm': 0.20552675214328775,
  'a_amp_norm': 0.08976636599379373,
  'freq': 111.51122457325411,
  'amp': 2724.4159149844845,
  'sgis': 834.3650216305031,
  'rms': 494.91472858269697},
 {'pd': 1.0,
  's0': 0.5720837116241455,
  's1': 0.5355679392814636,
  's2': 0.4838397800922394,
  's3': 0.46840256452560425,
  'a_freq_norm': -0.15269040132219058,
  'a_amp_norm': 0.29178822613331223,
  'freq': 78.37613787769737,
  'amp': 3229.470565333281,
  'sgis': 745.0323239928284,
  'rms': 467.99474652973964},
 {'pd': 1.0,
  's0': 0.6646009683609009,
  's1':

## 3b. Reload saved data (skip collection if already done)

In [4]:
datasets = {}
for p in sorted(Path("datasets").glob("pd*_n*.json")):
    key = p.stem.split("_")[0]
    datasets[key] = json.loads(p.read_text())
    print(f"{key}: {len(datasets[key])} samples")

pd0.3: 400 samples
pd0.6: 400 samples
pd1.0: 400 samples


## 4. Fit every architecture

Seconds per model. Val R2 is the number that matters — how much of the reward variance
the model explains on held-out data.

In [5]:
HIDDEN_DIMS = [16, 32, 64, 128]
SEEDS = [0, 1, 2]

models, fit_stats = {}, []
for name, rows in sorted(datasets.items()):
    print(f"\n{name}  (n={len(rows)})")
    for h in HIDDEN_DIMS:
        r2s = []
        for seed in SEEDS:
            m, st = rm.fit(rows, cfg, hidden_dim=h, seed=seed, verbose=False)
            r2s.append(st["val_r2"])
            if seed == 0:
                models[f"{name}_h{h}"] = m
                fit_stats.append({**st, "dataset": name})
        print(f"  hidden={h:<4} params={st['n_params']:<6} "
              f"val R2 {np.mean(r2s):.3f} +/- {np.std(r2s):.3f}")


pd0.3  (n=400)
  hidden=16   params=401    val R2 0.445 +/- 0.121
  hidden=32   params=1313   val R2 0.457 +/- 0.118
  hidden=64   params=4673   val R2 0.414 +/- 0.113
  hidden=128  params=17537  val R2 0.443 +/- 0.120

pd0.6  (n=400)
  hidden=16   params=401    val R2 0.474 +/- 0.029
  hidden=32   params=1313   val R2 0.496 +/- 0.043
  hidden=64   params=4673   val R2 0.469 +/- 0.020
  hidden=128  params=17537  val R2 0.493 +/- 0.021

pd1.0  (n=400)
  hidden=16   params=401    val R2 0.494 +/- 0.066
  hidden=32   params=1313   val R2 0.497 +/- 0.068
  hidden=64   params=4673   val R2 0.506 +/- 0.064
  hidden=128  params=17537  val R2 0.514 +/- 0.064


## 5. Cross-evaluation matrix

**This is the deliverable.** Each model scored against every severity's data.

- Strong diagonal, weak off-diagonal -> specialization is real, a selector is worth building
- Flat everywhere -> one model generalizes, no selector needed

In [6]:
for h in HIDDEN_DIMS:
    sub = {k.replace(f"_h{h}", ""): v for k, v in models.items() if k.endswith(f"_h{h}")}
    if len(sub) < 2:
        continue
    print(f"\n=== hidden={h} ===")
    rm.cross_evaluate(sub, datasets, cfg)


=== hidden=16 ===

model \ data             pd0.3         pd0.6         pd1.0
----------------------------------------------------------
pd0.3                    0.487         0.244         0.049
pd0.6                    0.213         0.489         0.327
pd1.0                   -0.038         0.294         0.473

(values are val R2 — how well each model predicts each severity's rewards)

=== hidden=32 ===

model \ data             pd0.3         pd0.6         pd1.0
----------------------------------------------------------
pd0.3                    0.497         0.243         0.051
pd0.6                    0.250         0.523         0.366
pd1.0                   -0.057         0.294         0.570

(values are val R2 — how well each model predicts each severity's rewards)

=== hidden=64 ===

model \ data             pd0.3         pd0.6         pd1.0
----------------------------------------------------------
pd0.3                    0.495         0.245         0.069
pd0.6                

## 6. Which action does each model choose

If the models pick genuinely different actions by severity, that is the mechanism a
selector would exploit. If they all pick the same one, specialization is not actionable
even if the R2 matrix looks specialized.

In [7]:
states = [np.array([r[f"s{i}"] for i in range(cfg.state_dim)], dtype=np.float32)
          for r in datasets[f"pd{PD_VALUES[1]}"][:60]]

for h in HIDDEN_DIMS:
    sub = {k.replace(f"_h{h}", ""): v for k, v in models.items() if k.endswith(f"_h{h}")}
    if len(sub) < 2:
        continue
    print(f"\n=== hidden={h} ===")
    rm.action_agreement(sub, cfg, states)


=== hidden=16 ===
pd0.3            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd0.6            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd1.0            picks 180Hz/   0uA in 100% of states (1 distinct actions)

=== hidden=32 ===
pd0.3            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd0.6            picks 180Hz/   0uA in 68% of states (2 distinct actions)
pd1.0            picks 180Hz/   0uA in 100% of states (1 distinct actions)

=== hidden=64 ===
pd0.3            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd0.6            picks 180Hz/1000uA in 60% of states (3 distinct actions)
pd1.0            picks 180Hz/   0uA in 100% of states (1 distinct actions)

=== hidden=128 ===
pd0.3            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd0.6            picks 180Hz/   0uA in 100% of states (1 distinct actions)
pd1.0            picks 180Hz/   0uA in 100% of states (1 distinct actions)


In [8]:
import numpy as np

for name, rows in sorted(datasets.items()):
    print(f"\n=== {name} ===")
    print(f"{'amp bin':<14} {'n':>4} {'mean sgis':>10} {'mean reward':>12}")
    bins = [(0, 500), (500, 1500), (1500, 2500), (2500, 3500), (3500, 5000)]
    for lo, hi in bins:
        sel = [r for r in rows if lo <= r["amp"] < hi]
        if not sel:
            continue
        r1 = [np.clip((r["sgis"] - cfg.sgis_min) / (cfg.sgis_max - cfg.sgis_min), 0, 1) for r in sel]
        r2 = [np.clip(r["rms"] / cfg.rms_max, 0, 1) for r in sel]
        rew = [-cfg.w_biomarker * a - cfg.w_energy * b for a, b in zip(r1, r2)]
        print(f"{lo:>5}-{hi:<8} {len(sel):>4} {np.mean([r['sgis'] for r in sel]):>10.0f} "
              f"{np.mean(rew):>12.4f}")


=== pd0.3 ===
amp bin           n  mean sgis  mean reward
    0-500        46        714      -0.1801
  500-1500       85        792      -0.2309
 1500-2500       86        786      -0.2530
 2500-3500       71        717      -0.2540
 3500-5000      112        716      -0.2886

=== pd0.6 ===
amp bin           n  mean sgis  mean reward
    0-500        46        718      -0.1818
  500-1500       85        833      -0.2477
 1500-2500       86        832      -0.2720
 2500-3500       71        787      -0.2835
 3500-5000      112        756      -0.3053

=== pd1.0 ===
amp bin           n  mean sgis  mean reward
    0-500        46        734      -0.1885
  500-1500       85        888      -0.2706
 1500-2500       86        871      -0.2881
 2500-3500       71        825      -0.2992
 3500-5000      112        827      -0.3343


In [9]:
import numpy as np

fbins = [(0, 60), (60, 110), (110, 145), (145, 168), (168, 186)]
abins = [(0, 500), (500, 1500), (1500, 3000), (3000, 5000)]

for name, rows in sorted(datasets.items()):
    print(f"\n=== {name} — mean sgis ===")
    print(f"{'freq':<12}" + "".join(f"{f'{lo}-{hi}':>12}" for lo, hi in abins))
    for flo, fhi in fbins:
        line = f"{f'{flo}-{fhi}Hz':<12}"
        for alo, ahi in abins:
            sel = [r for r in rows if flo <= r["freq"] < fhi and alo <= r["amp"] < ahi]
            line += f"{np.mean([r['sgis'] for r in sel]):>12.0f}" if len(sel) >= 3 else f"{'—':>12}"
        print(line)


=== pd0.3 — mean sgis ===
freq               0-500    500-1500   1500-3000   3000-5000
0-60Hz               870         963         934         922
60-110Hz             740         765         730         730
110-145Hz            697         723         750         662
145-168Hz            597         622         542         471
168-186Hz            566         418         453         450

=== pd0.6 — mean sgis ===
freq               0-500    500-1500   1500-3000   3000-5000
0-60Hz               737         885         895         856
60-110Hz             768         905         881         858
110-145Hz            700         867         857         748
145-168Hz            678         625         656         562
168-186Hz            705         412         440         471

=== pd1.0 — mean sgis ===
freq               0-500    500-1500   1500-3000   3000-5000
0-60Hz               762         971         934         888
60-110Hz             768         962         942         938
110-

## 7. Validate in MATLAB

The model's chosen action, run for real, against the fixed baselines. This is what says
whether the reward model beats both TD3 and open-loop DBS.

In [ ]:
for pd in PD_VALUES:
    c = replace(cfg, pd_value=pd)
    menv.cfg = c
    model = models[f"pd{pd}_h64"]
    s = menv.reset()
    (freq, amp), pred, _ = rm.best_action(model, s, c)
    print(f"\npd={pd}: model picks {freq}Hz/{amp}uA (predicted reward {pred:+.4f})")
    fixed = dbs_checks.baseline(menv, c, actions=[(0, 0), (freq, amp), (180, 1000)],
                                n_episodes=3)

## Shut down

In [ ]:
menv.close()
print("engine closed")